# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col, upper, lower, trim, substring
from pyspark.sql.types import StringType, IntegerType, DateType

# Read the data from bronze table 

In [0]:
df = spark.table("workspace.bronze_schema.crm_cust_info")

# Explore dataframe 

In [0]:
df.display() # display the crm_cust_info dataframe read from the bronze_schema

In [0]:
df.printSchema() #Check datatypes and nullable columns

In [0]:
df.schema.fields

In [0]:
df.describe().display()
"""
View more information about the data, count, mean etc
"""

## Checking cst_id for duplicates and null values

In [0]:
# Checking cst_id for duplicates and null values 
df.select("*").groupBy("cst_id").count().where((col("count") > 1) | col("cst_id").isNull()).display()

In [0]:
df.select(col("cst_marital_status")).distinct().display() # Checking marital status options 

In [0]:
df.select(col("cst_gndr")).distinct().display()  # Checking the available gender options

## Checking for whitespaces

In [0]:
# Checking for white spaces
for column in df.columns:
    white_space_df = df.select(column).filter(col(column).startswith(" ") | col(column).endswith(" "))
    white_space_count = white_space_df.count()

    if white_space_count:
        print(f"White space detected in Column: {column}")
        white_space_df.display()
    else: print(f"No white space detected in Column: {column}")

In [0]:
df.columns # View column names

## Checking for null values

In [0]:
# Checking for null values in each column
for column in df.columns:
    null_df = df.select("*").filter(col(column).isNull())
    
    null_count = null_df.count()

    if null_count:
        print(f"Null values detected in Column: {column}")
        print(f"Null count: {null_count}")
        null_df.display()
    else:
        print(f"No null value detected in Columns: {column}")

# Data Note

- Null values and duplicate values found in the cst_id column
- Nullable values are allowed. Might need to replace with n/a 
- All datatypes are as supposed for their respective columns.
- From the describe count, we can see that some values are missing.
- cst_marital_status and cst_gndr both have 3 possible values, however name a abbreviated and normalization needs to be done.
- White spaces present in columns cst_firstname and cst_lastname
- Column names are not well formatted. column rename needed
-  All columns have null values except from the cst_id column

## Conclusion: Data is unclean and needs Transformation 

# Data Transformation

## Cleaning Null and duplicate values in the cst_id column

In [0]:
df = (
  df.select(
    "*",
    F.row_number()
    .over(Window.partitionBy("cst_id")
    .orderBy(col("cst_create_date").desc()))
    .alias("row_number")
  ).where((col("row_number") == 1))
)




In [0]:
# Display the deduplicated DataFrame after dropping the ambiguous row_number column

df = df.drop('row_number')
df.display()

## Triming white spaces

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
else: 
    print("Trimming Completed...")

df.display()

## Normalizing Columns 


###### cst_marital_status column
- M: Married
- S: Single
###### cst_gndr and 
- M: Male
- F: Female

In [0]:

df = df.withColumn(
    "cst_gndr", 
    F.when(upper(col("cst_gndr")) == "M", "Male")
    .when(upper(col("cst_gndr")) == "F", "Female")
    .otherwise("n/a")
).withColumn(
    "cst_marital_status",
    F.when(upper(col("cst_marital_status")) == "M", "Married")
    .when (upper(col("cst_marital_status")) == "S", "Single")
    .otherwise("n/a")
)

In [0]:
df.display()

## Formatting column names

In [0]:
# Create a dictionary with old name as key and new name as value
COLUMN_RENAME_MAP = {
    'cst_id': "customer_id",
    'cst_key': "customer_key",
    'cst_firstname': "customer_firstname",
    'cst_lastname': "customer_lastname",
    'cst_marital_status': "customer_marital_status",
    'cst_gndr': "customer_gender",
    'cst_create_date': "customer_create_date"
}

print(type(COLUMN_RENAME_MAP))

for key, value in COLUMN_RENAME_MAP.items():
    print(f"Formatting {key} to {value} ...")
    df = df.withColumnRenamed(key, value)
    print(f"Formatting completed")

df.display()

# Load data into silver schema

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver_schema.crm_cust_info")

## View table from the silver schema

In [0]:
silver_crm_cust_info = spark.table("workspace.silver_schema.crm_cust_info")
silver_crm_cust_info.display()